In [1]:
from collections import OrderedDict
import pandas as pd

In [11]:
# ============================================================
# Raw layer-wise pruning stats
# ============================================================

DATA = OrderedDict({
    "20%": OrderedDict({
        "features.0":   {"zeros": 7,        "total": 1728},
        "features.2":   {"zeros": 1081,     "total": 36864},
        "features.5":   {"zeros": 2228,     "total": 73728},
        "features.7":   {"zeros": 5138,     "total": 147456},
        "features.10":  {"zeros": 11816,    "total": 294912},
        "features.12":  {"zeros": 29149,    "total": 589824},
        "features.14":  {"zeros": 28450,    "total": 589824},
        "features.17":  {"zeros": 67539,    "total": 1179648},
        "features.19":  {"zeros": 169014,   "total": 2359296},
        "features.21":  {"zeros": 168863,   "total": 2359296},
        "features.24":  {"zeros": 158506,   "total": 2359296},
        "features.26":  {"zeros": 157673,   "total": 2359296},
        "features.28":  {"zeros": 177098,   "total": 2359296},
        "classifier.0": {"zeros": 23945866, "total": 102760448},
        "classifier.3": {"zeros": 1932135,  "total": 16777216},
        "classifier.6": {"zeros": 3255,     "total": 40960},
    }),
    "40%": OrderedDict({
        "features.0":   {"zeros": 21,       "total": 1728},
        "features.2":   {"zeros": 2268,     "total": 36864},
        "features.5":   {"zeros": 4640,     "total": 73728},
        "features.7":   {"zeros": 10673,    "total": 147456},
        "features.10":  {"zeros": 25239,    "total": 294912},
        "features.12":  {"zeros": 60999,    "total": 589824},
        "features.14":  {"zeros": 59848,    "total": 589824},
        "features.17":  {"zeros": 142060,   "total": 1179648},
        "features.19":  {"zeros": 354532,   "total": 2359296},
        "features.21":  {"zeros": 354568,   "total": 2359296},
        "features.24":  {"zeros": 332611,   "total": 2359296},
        "features.26":  {"zeros": 330446,   "total": 2359296},
        "features.28":  {"zeros": 370371,   "total": 2359296},
        "classifier.0": {"zeros": 47667071, "total": 102760448},
        "classifier.3": {"zeros": 3993353,  "total": 16777216},
        "classifier.6": {"zeros": 6935,     "total": 40960},
    }),
    "60%": OrderedDict({
        "features.0":   {"zeros": 38,       "total": 1728},
        "features.2":   {"zeros": 3682,     "total": 36864},
        "features.5":   {"zeros": 7663,     "total": 73728},
        "features.7":   {"zeros": 17852,    "total": 147456},
        "features.10":  {"zeros": 41928,    "total": 294912},
        "features.12":  {"zeros": 101090,   "total": 589824},
        "features.14":  {"zeros": 99250,    "total": 589824},
        "features.17":  {"zeros": 234522,   "total": 1179648},
        "features.19":  {"zeros": 584120,   "total": 2359296},
        "features.21":  {"zeros": 583638,   "total": 2359296},
        "features.24":  {"zeros": 547764,   "total": 2359296},
        "features.26":  {"zeros": 545263,   "total": 2359296},
        "features.28":  {"zeros": 607530,   "total": 2359296},
        "classifier.0": {"zeros": 70773910, "total": 102760448},
        "classifier.3": {"zeros": 6413753,  "total": 16777216},
        "classifier.6": {"zeros": 11450,    "total": 40960},
    }),
    "80%": OrderedDict({
        "features.0":   {"zeros": 68,       "total": 1728},
        "features.2":   {"zeros": 6043,     "total": 36864},
        "features.5":   {"zeros": 12604,    "total": 73728},
        "features.7":   {"zeros": 29311,    "total": 147456},
        "features.10":  {"zeros": 68456,    "total": 294912},
        "features.12":  {"zeros": 165538,   "total": 589824},
        "features.14":  {"zeros": 162502,   "total": 589824},
        "features.17":  {"zeros": 381736,   "total": 1179648},
        "features.19":  {"zeros": 942114,   "total": 2359296},
        "features.21":  {"zeros": 943486,   "total": 2359296},
        "features.24":  {"zeros": 885674,   "total": 2359296},
        "features.26":  {"zeros": 882653,   "total": 2359296},
        "features.28":  {"zeros": 970959,   "total": 2359296},
        "classifier.0": {"zeros": 92092848, "total": 102760448},
        "classifier.3": {"zeros": 9868991,  "total": 16777216},
        "classifier.6": {"zeros": 18287,    "total": 40960},
    }),
})

# ============================================================
# Region definitions
# ============================================================

REGIONS = OrderedDict({
    "Early conv": ["features.0", "features.2", "features.5", "features.7"],
    "Mid conv":   ["features.10", "features.12", "features.14"],
    "Late conv":  ["features.17", "features.19", "features.21", "features.24", "features.26", "features.28"],
    "Classifier": ["classifier.0", "classifier.3", "classifier.6"],
})

REGIONS_2WAY = OrderedDict({
    "All features": [k for k in DATA["20%"].keys() if k.startswith("features.")],
    "Classifier":   [k for k in DATA["20%"].keys() if k.startswith("classifier.")],
})


In [4]:

# ============================================================
# Helpers
# ============================================================

def layer_sparsity(layer_stats):
    return layer_stats["zeros"] / layer_stats["total"]

def weighted_region_sparsity(pruning_stats, layers):
    zeros = sum(pruning_stats[layer]["zeros"] for layer in layers)
    total = sum(pruning_stats[layer]["total"] for layer in layers)
    return zeros / total

def unweighted_region_sparsity(pruning_stats, layers):
    vals = [layer_sparsity(pruning_stats[layer]) for layer in layers]
    return sum(vals) / len(vals)

def model_total_params(pruning_stats):
    return sum(v["total"] for v in pruning_stats.values())

def model_total_zeros(pruning_stats):
    return sum(v["zeros"] for v in pruning_stats.values())

def region_param_share(pruning_stats, layers):
    total_region = sum(pruning_stats[layer]["total"] for layer in layers)
    total_model = model_total_params(pruning_stats)
    return total_region / total_model

def pruning_concentration(pruning_stats, layers):
    zeros_region = sum(pruning_stats[layer]["zeros"] for layer in layers)
    zeros_model = model_total_zeros(pruning_stats)
    return zeros_region / zeros_model

def as_percent(x, digits=2):
    return f"{100*x:.{digits}f}%"

def make_table_from_region_metric(data, regions, metric_fn, digits=2):
    rows = []
    for region_name, layers in regions.items():
        row = {"Region": region_name}
        for sparsity_name, pruning_stats in data.items():
            row[sparsity_name] = as_percent(metric_fn(pruning_stats, layers), digits)
        rows.append(row)
    return pd.DataFrame(rows)

def make_parameter_share_table(reference_stats, regions, digits=2):
    rows = []
    for region_name, layers in regions.items():
        rows.append({
            "Region": region_name,
            "Parameter Share of Total Model": as_percent(
                region_param_share(reference_stats, layers), digits
            )
        })
    return pd.DataFrame(rows)

def print_tsv(df, title):
    print(f"\n{title}")
    print(df.to_csv(sep="\t", index=False))



In [5]:
# ============================================================
# Generate the 5 requested tables
# ============================================================

# 1) Weighted pruning by model region
weighted_region_df = make_table_from_region_metric(
    DATA, REGIONS, weighted_region_sparsity, digits=2
)

# 2) Unweighted average layer sparsity by region
unweighted_region_df = make_table_from_region_metric(
    DATA, REGIONS, unweighted_region_sparsity, digits=2
)

# 3) Parameter share of each region
# totals are identical across pruning levels, so use any one reference block
reference_stats = next(iter(DATA.values()))
parameter_share_df = make_parameter_share_table(reference_stats, REGIONS, digits=2)

# 4) Conv vs classifier only
conv_vs_classifier_df = make_table_from_region_metric(
    DATA, REGIONS_2WAY, weighted_region_sparsity, digits=2
)

# 5) Pruning concentration table
pruning_concentration_df = make_table_from_region_metric(
    DATA, REGIONS, pruning_concentration, digits=2
)

# ============================================================
# Driver code: display / export
# ============================================================

if __name__ == "__main__":
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)

    print_tsv(weighted_region_df, "Weighted pruning by model region")
    print_tsv(unweighted_region_df, "Unweighted average layer sparsity by region")
    print_tsv(parameter_share_df, "Parameter share of each region")
    print_tsv(conv_vs_classifier_df, "Conv vs classifier only")
    print_tsv(pruning_concentration_df, "Pruning concentration")

    # Optional: save to CSV files
    weighted_region_df.to_csv("weighted_pruning_by_region.csv", index=False)
    unweighted_region_df.to_csv("unweighted_avg_layer_sparsity_by_region.csv", index=False)
    parameter_share_df.to_csv("parameter_share_by_region.csv", index=False)
    conv_vs_classifier_df.to_csv("conv_vs_classifier.csv", index=False)
    pruning_concentration_df.to_csv("pruning_concentration.csv", index=False)


Weighted pruning by model region
Region	20%	40%	60%	80%
Early conv	3.25%	6.78%	11.25%	18.49%
Mid conv	4.71%	9.91%	16.43%	26.89%
Late conv	6.93%	14.52%	23.91%	38.58%
Classifier	21.64%	43.21%	64.56%	85.28%


Unweighted average layer sparsity by region
Region	20%	40%	60%	80%
Early conv	2.46%	5.22%	8.67%	14.33%
Mid conv	4.59%	9.68%	16.06%	26.28%
Late conv	6.83%	14.32%	23.58%	38.06%
Classifier	14.26%	29.04%	45.02%	64.36%


Parameter share of each region
Region	Parameter Share of Total Model
Early conv	0.19%
Mid conv	1.10%
Late conv	9.66%
Classifier	89.05%


Conv vs classifier only
Region	20%	40%	60%	80%
All features	6.64%	13.92%	22.94%	37.06%
Classifier	21.64%	43.21%	64.56%	85.28%


Pruning concentration
Region	20%	40%	60%	80%
Early conv	0.03%	0.03%	0.04%	0.04%
Mid conv	0.26%	0.27%	0.30%	0.37%
Late conv	3.35%	3.51%	3.85%	4.66%
Classifier	96.36%	96.19%	95.81%	94.93%



In [10]:
from collections import OrderedDict
import pandas as pd

# ============================================================
# This driver is filled with the exact stats you pasted
# ============================================================

DATA = OrderedDict({
    "20%": OrderedDict({
        "features.0":   {"zeros": 12,       "total": 1728},
        "features.2":   {"zeros": 1081,     "total": 36864},
        "features.5":   {"zeros": 2259,     "total": 73728},
        "features.7":   {"zeros": 5153,     "total": 147456},
        "features.10":  {"zeros": 11888,    "total": 294912},
        "features.12":  {"zeros": 29155,    "total": 589824},
        "features.14":  {"zeros": 28558,    "total": 589824},
        "features.17":  {"zeros": 67819,    "total": 1179648},
        "features.19":  {"zeros": 169221,   "total": 2359296},
        "features.21":  {"zeros": 169379,   "total": 2359296},
        "features.24":  {"zeros": 158997,   "total": 2359296},
        "features.26":  {"zeros": 158191,   "total": 2359296},
        "features.28":  {"zeros": 177737,   "total": 2359296},
        "classifier.0": {"zeros": 23983254, "total": 102760448},
        "classifier.3": {"zeros": 1932216,  "total": 16777216},
        "classifier.6": {"zeros": 36626,    "total": 409600},
    }),
    "40%": OrderedDict({
        "features.0":   {"zeros": 22,       "total": 1728},
        "features.2":   {"zeros": 2274,     "total": 36864},
        "features.5":   {"zeros": 4644,     "total": 73728},
        "features.7":   {"zeros": 10741,    "total": 147456},
        "features.10":  {"zeros": 25295,    "total": 294912},
        "features.12":  {"zeros": 61231,    "total": 589824},
        "features.14":  {"zeros": 60025,    "total": 589824},
        "features.17":  {"zeros": 142424,   "total": 1179648},
        "features.19":  {"zeros": 355570,   "total": 2359296},
        "features.21":  {"zeros": 355758,   "total": 2359296},
        "features.24":  {"zeros": 333782,   "total": 2359296},
        "features.26":  {"zeros": 331546,   "total": 2359296},
        "features.28":  {"zeros": 371548,   "total": 2359296},
        "classifier.0": {"zeros": 47739751, "total": 102760448},
        "classifier.3": {"zeros": 3991886,  "total": 16777216},
        "classifier.6": {"zeros": 76594,    "total": 409600},
    }),
    "60%": OrderedDict({
        "features.0":   {"zeros": 38,       "total": 1728},
        "features.2":   {"zeros": 3696,     "total": 36864},
        "features.5":   {"zeros": 7634,     "total": 73728},
        "features.7":   {"zeros": 17970,    "total": 147456},
        "features.10":  {"zeros": 41997,    "total": 294912},
        "features.12":  {"zeros": 101518,   "total": 589824},
        "features.14":  {"zeros": 99560,    "total": 589824},
        "features.17":  {"zeros": 235240,   "total": 1179648},
        "features.19":  {"zeros": 586415,   "total": 2359296},
        "features.21":  {"zeros": 585848,   "total": 2359296},
        "features.24":  {"zeros": 549623,   "total": 2359296},
        "features.26":  {"zeros": 546885,   "total": 2359296},
        "features.28":  {"zeros": 609706,   "total": 2359296},
        "classifier.0": {"zeros": 70866010, "total": 102760448},
        "classifier.3": {"zeros": 6416247,  "total": 16777216},
        "classifier.6": {"zeros": 126250,   "total": 409600},
    }),
    "80%": OrderedDict({
        "features.0":   {"zeros": 70,       "total": 1728},
        "features.2":   {"zeros": 6043,     "total": 36864},
        "features.5":   {"zeros": 12701,    "total": 73728},
        "features.7":   {"zeros": 29419,    "total": 147456},
        "features.10":  {"zeros": 68735,    "total": 294912},
        "features.12":  {"zeros": 166162,   "total": 589824},
        "features.14":  {"zeros": 163186,   "total": 589824},
        "features.17":  {"zeros": 383213,   "total": 1179648},
        "features.19":  {"zeros": 945612,   "total": 2359296},
        "features.21":  {"zeros": 947222,   "total": 2359296},
        "features.24":  {"zeros": 888982,   "total": 2359296},
        "features.26":  {"zeros": 886160,   "total": 2359296},
        "features.28":  {"zeros": 974394,   "total": 2359296},
        "classifier.0": {"zeros": 92171477, "total": 102760448},
        "classifier.3": {"zeros": 9877909,  "total": 16777216},
        "classifier.6": {"zeros": 204897,   "total": 409600},
    }),
})

PERF = OrderedDict({
    "20%": {
        "global_sparsity": 0.2000,
        "pruning_only_val_loss": 0.8848,
        "pruning_only_val_acc": 73.30,
        "final_val_loss": 0.8669,
        "final_val_acc": 73.74,
        "final_test_loss": 0.9003,
        "final_test_acc": 72.84,
    },
    "40%": {
        "global_sparsity": 0.4000,
        "pruning_only_val_loss": 0.8881,
        "pruning_only_val_acc": 72.82,
        "final_val_loss": 0.8642,
        "final_val_acc": 73.66,
        "final_test_loss": 0.8920,
        "final_test_acc": 73.37,
    },
    "60%": {
        "global_sparsity": 0.6000,
        "pruning_only_val_loss": 0.9209,
        "pruning_only_val_acc": 72.12,
        "final_val_loss": 0.8686,
        "final_val_acc": 73.52,
        "final_test_loss": 0.8940,
        "final_test_acc": 73.08,
    },
    "80%": {
        "global_sparsity": 0.8000,
        "pruning_only_val_loss": 1.4331,
        "pruning_only_val_acc": 69.46,
        "final_val_loss": 0.9005,
        "final_val_acc": 72.90,
        "final_test_loss": 0.9256,
        "final_test_acc": 72.45,
    },
})

REGIONS = OrderedDict({
    "Early conv": ["features.0", "features.2", "features.5", "features.7"],
    "Mid conv":   ["features.10", "features.12", "features.14"],
    "Late conv":  ["features.17", "features.19", "features.21", "features.24", "features.26", "features.28"],
    "Classifier": ["classifier.0", "classifier.3", "classifier.6"],
})

REGIONS_2WAY = OrderedDict({
    "All features": [k for k in DATA["20%"].keys() if k.startswith("features.")],
    "Classifier":   [k for k in DATA["20%"].keys() if k.startswith("classifier.")],
})

def layer_sparsity(layer_stats):
    return layer_stats["zeros"] / layer_stats["total"]

def weighted_region_sparsity(pruning_stats, layers):
    zeros = sum(pruning_stats[layer]["zeros"] for layer in layers)
    total = sum(pruning_stats[layer]["total"] for layer in layers)
    return zeros / total

def unweighted_region_sparsity(pruning_stats, layers):
    vals = [layer_sparsity(pruning_stats[layer]) for layer in layers]
    return sum(vals) / len(vals)

def model_total_params(pruning_stats):
    return sum(v["total"] for v in pruning_stats.values())

def model_total_zeros(pruning_stats):
    return sum(v["zeros"] for v in pruning_stats.values())

def region_param_share(pruning_stats, layers):
    total_region = sum(pruning_stats[layer]["total"] for layer in layers)
    total_model = model_total_params(pruning_stats)
    return total_region / total_model

def pruning_concentration(pruning_stats, layers):
    zeros_region = sum(pruning_stats[layer]["zeros"] for layer in layers)
    zeros_model = model_total_zeros(pruning_stats)
    return zeros_region / zeros_model

def as_percent(x, digits=2):
    return f"{100*x:.{digits}f}%"

def make_table_from_region_metric(data, regions, metric_fn, digits=2):
    rows = []
    for region_name, layers in regions.items():
        row = {"Region": region_name}
        for sparsity_name, pruning_stats in data.items():
            row[sparsity_name] = as_percent(metric_fn(pruning_stats, layers), digits)
        rows.append(row)
    return pd.DataFrame(rows)

def make_parameter_share_table(reference_stats, regions, digits=2):
    rows = []
    for region_name, layers in regions.items():
        rows.append({
            "Region": region_name,
            "Parameter Share of Total Model": as_percent(
                region_param_share(reference_stats, layers), digits
            )
        })
    return pd.DataFrame(rows)

def make_performance_summary(perf):
    rows = []
    for pruning_level, stats in perf.items():
        rows.append({
            "Pruning Level": pruning_level,
            "Global Sparsity": f"{stats['global_sparsity']:.4f}",
            "Pruning-Only Val Loss": f"{stats['pruning_only_val_loss']:.4f}",
            "Pruning-Only Val Acc": f"{stats['pruning_only_val_acc']:.2f}%",
            "Final Val Loss": f"{stats['final_val_loss']:.4f}",
            "Final Val Acc": f"{stats['final_val_acc']:.2f}%",
            "Final Test Loss": f"{stats['final_test_loss']:.4f}",
            "Final Test Acc": f"{stats['final_test_acc']:.2f}%",
        })
    return pd.DataFrame(rows)

def make_recovery_table(perf):
    rows = []
    for pruning_level, stats in perf.items():
        gain = stats["final_val_acc"] - stats["pruning_only_val_acc"]
        rows.append({
            "Pruning Level": pruning_level,
            "Val Acc After Pruning": f"{stats['pruning_only_val_acc']:.2f}%",
            "Val Acc After FT": f"{stats['final_val_acc']:.2f}%",
            "Recovery": f"{gain:+.2f}%",
        })
    return pd.DataFrame(rows)

def print_tsv(df, title):
    print(f"\n{title}")
    print(df.to_csv(sep="\t", index=False))

if __name__ == "__main__":
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)

    weighted_region_df = make_table_from_region_metric(
        DATA, REGIONS, weighted_region_sparsity, digits=2
    )

    unweighted_region_df = make_table_from_region_metric(
        DATA, REGIONS, unweighted_region_sparsity, digits=2
    )

    parameter_share_df = make_parameter_share_table(
        next(iter(DATA.values())), REGIONS, digits=2
    )

    conv_vs_classifier_df = make_table_from_region_metric(
        DATA, REGIONS_2WAY, weighted_region_sparsity, digits=2
    )

    pruning_concentration_df = make_table_from_region_metric(
        DATA, REGIONS, pruning_concentration, digits=2
    )

    performance_summary_df = make_performance_summary(PERF)
    recovery_df = make_recovery_table(PERF)

    print_tsv(weighted_region_df, "Weighted pruning by model region")
    print_tsv(unweighted_region_df, "Unweighted average layer sparsity by region")
    print_tsv(parameter_share_df, "Parameter share of each region")
    print_tsv(conv_vs_classifier_df, "Conv vs classifier only")
    print_tsv(pruning_concentration_df, "Pruning concentration")
    print_tsv(performance_summary_df, "Overall performance summary")
    print_tsv(recovery_df, "Fine-tuning recovery")

    weighted_region_df.to_csv("weighted_pruning_by_region.csv", index=False)
    unweighted_region_df.to_csv("unweighted_avg_layer_sparsity_by_region.csv", index=False)
    parameter_share_df.to_csv("parameter_share_by_region.csv", index=False)
    conv_vs_classifier_df.to_csv("conv_vs_classifier.csv", index=False)
    pruning_concentration_df.to_csv("pruning_concentration.csv", index=False)
    performance_summary_df.to_csv("performance_summary.csv", index=False)
    recovery_df.to_csv("fine_tuning_recovery.csv", index=False)


Weighted pruning by model region
Region	20%	40%	60%	80%
Early conv	3.27%	6.81%	11.29%	18.57%
Mid conv	4.72%	9.94%	16.48%	27.00%
Late conv	6.95%	14.57%	24.00%	38.73%
Classifier	21.64%	43.19%	64.54%	85.25%


Unweighted average layer sparsity by region
Region	20%	40%	60%	80%
Early conv	2.55%	5.26%	8.69%	14.41%
Mid conv	4.61%	9.71%	16.11%	26.38%
Late conv	6.85%	14.36%	23.66%	38.21%
Classifier	14.60%	29.65%	46.01%	66.20%


Parameter share of each region
Region	Parameter Share of Total Model
Early conv	0.19%
Mid conv	1.10%
Late conv	9.64%
Classifier	89.08%


Conv vs classifier only
Region	20%	40%	60%	80%
All features	6.66%	13.97%	23.02%	37.20%
Classifier	21.64%	43.19%	64.54%	85.25%


Pruning concentration
Region	20%	40%	60%	80%
Early conv	0.03%	0.03%	0.04%	0.04%
Mid conv	0.26%	0.27%	0.30%	0.37%
Late conv	3.35%	3.51%	3.85%	4.67%
Classifier	96.36%	96.19%	95.81%	94.92%


Overall performance summary
Pruning Level	Global Sparsity	Pruning-Only Val Loss	Pruning-Only Val Acc	Final Val Loss	Final Va

In [15]:
import re
from pathlib import Path
from collections import defaultdict, OrderedDict
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

# Set this to the folder containing your FGSM text files
LOG_DIR = Path(".")

OUTPUT_DIR = Path("./fgsm_tables_output")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Expected epsilons in desired column order
EPS_ORDER = [0.0078, 0.0156, 0.0313, 0.0627]

# Model row order
MODEL_ORDER = ["Dense", "Pruned 20%", "Pruned 40%", "Pruned 60%", "Pruned 80%"]

# ============================================================
# FILENAME PARSING
# ============================================================

def parse_filename(path: Path):
    """
    Supports names like:
      FGSM_finetuned_VGG_10.txt
      FGSM_finetuned_VGG_100.txt
      FGSM_prune_finetune_20_VGG_10.txt
      FGSM_prune_finetune20_VGG_100.txt
    """
    name = path.name

    # Dataset
    m_dataset = re.search(r"_VGG_(10|100)\.txt$", name)
    if not m_dataset:
        raise ValueError(f"Could not infer dataset from filename: {name}")
    dataset = f"CIFAR-{m_dataset.group(1)}"

    # Dense vs pruned
    if "FGSM_finetuned" in name:
        model_variant = "Dense"
        sparsity = None
    else:
        m_prune = re.search(r"prune_finetune_?(\d+)_VGG_", name)
        if not m_prune:
            raise ValueError(f"Could not infer pruning level from filename: {name}")
        sparsity = int(m_prune.group(1))
        model_variant = f"Pruned {sparsity}%"

    print(f"Dataset: {dataset}")

    return dataset, model_variant, sparsity


# ============================================================
# LOG PARSING
# ============================================================

BLOCK_RE = re.compile(
    r"Epsilon:\s*([0-9.]+)\s*"
    r".*?examples_10_count:\s*(\d+)\s*"
    r".*?examples_01_count:\s*(\d+)\s*"
    r".*?examples_000_count:\s*(\d+)\s*"
    r".*?examples_001_count:\s*(\d+)\s*"
    r".*?examples_11_count:\s*(\d+)\s*"
    r".*?Clean accuracy:\s*([0-9.]+)\s*"
    r".*?FGSM accuracy\s*\(epsilon=.*?\):\s*([0-9.]+)",
    re.DOTALL
)

def rounded_eps(x: float) -> float:
    return round(x + 1e-12, 4)

def parse_log_file(path: Path):
    dataset, model_variant, sparsity = parse_filename(path)
    text = path.read_text(encoding="utf-8", errors="ignore")

    matches = list(BLOCK_RE.finditer(text))
    if not matches:
        raise ValueError(f"No epsilon blocks found in {path.name}")

    records_by_eps = OrderedDict()

    for m in matches:
        eps = rounded_eps(float(m.group(1)))
        record = {
            "dataset": dataset,
            "model_variant": model_variant,
            "sparsity": sparsity,
            "epsilon": eps,
            "examples_10_count": int(m.group(2)),
            "examples_01_count": int(m.group(3)),
            "examples_000_count": int(m.group(4)),
            "examples_001_count": int(m.group(5)),
            "examples_11_count": int(m.group(6)),
            "clean_accuracy": float(m.group(7)),
            "fgsm_accuracy": float(m.group(8)),
            "source_file": path.name,
        }

        # Deduplicate repeated epsilon blocks within the same file.
        # If repeated blocks disagree, raise an error.
        if eps in records_by_eps:
            prev = records_by_eps[eps].copy()
            curr = record.copy()
            prev.pop("source_file", None)
            curr.pop("source_file", None)
            if prev != curr:
                raise ValueError(
                    f"Inconsistent duplicate epsilon block in {path.name} for epsilon={eps}\n"
                    f"Previous: {prev}\nCurrent: {curr}"
                )
        else:
            records_by_eps[eps] = record

    return list(records_by_eps.values())


# ============================================================
# CONSISTENCY CHECKS
# ============================================================

def validate_record(rec, total_examples=10000, tol=1e-3):
    c10 = rec["examples_10_count"]
    c01 = rec["examples_01_count"]
    c000 = rec["examples_000_count"]
    c001 = rec["examples_001_count"]
    c11 = rec["examples_11_count"]

    total = c10 + c01 + c000 + c001 + c11
    if total != total_examples:
        raise ValueError(
            f"Count total mismatch in {rec['source_file']} @ eps={rec['epsilon']}: "
            f"{total} != {total_examples}"
        )

    clean_acc_from_counts = (c10 + c11) / total_examples
    fgsm_acc_from_counts = (c01 + c11) / total_examples

    if abs(clean_acc_from_counts - rec["clean_accuracy"]) > tol:
        raise ValueError(
            f"Clean accuracy mismatch in {rec['source_file']} @ eps={rec['epsilon']}: "
            f"{clean_acc_from_counts:.4f} vs {rec['clean_accuracy']:.4f}"
        )

    if abs(fgsm_acc_from_counts - rec["fgsm_accuracy"]) > tol:
        raise ValueError(
            f"FGSM accuracy mismatch in {rec['source_file']} @ eps={rec['epsilon']}: "
            f"{fgsm_acc_from_counts:.4f} vs {rec['fgsm_accuracy']:.4f}"
        )

def validate_all(records):
    for rec in records:
        validate_record(rec)


# ============================================================
# TABLE BUILDERS
# ============================================================

def pivot_metric(df, dataset_name, metric_col):
    sub = df[df["dataset"] == dataset_name].copy()

    # Order model rows
    sub["model_variant"] = pd.Categorical(
        sub["model_variant"], categories=MODEL_ORDER, ordered=True
    )

    pt = sub.pivot_table(
        index="model_variant",
        columns="epsilon",
        values=metric_col,
        aggfunc="first",
    ).sort_index()

    # Reorder epsilon columns if present
    present_cols = [c for c in EPS_ORDER if c in pt.columns]
    pt = pt[present_cols]

    # Pretty column labels
    pt.columns = [f"{c:.4f}" for c in pt.columns]
    pt.index.name = "Model Variant"
    return pt.reset_index()

def save_table(df, base_name):
    csv_path = OUTPUT_DIR / f"{base_name}.csv"
    tsv_path = OUTPUT_DIR / f"{base_name}.tsv"
    df.to_csv(csv_path, index=False)
    df.to_csv(tsv_path, sep="\t", index=False)

def print_tsv(title, df):
    print(f"\n{title}")
    print(df.to_csv(sep="\t", index=False))

def build_all_tables(df, dataset_name):
    tables = OrderedDict()

    tables["clean_accuracy"] = pivot_metric(df, dataset_name, "clean_accuracy")
    tables["fgsm_accuracy"] = pivot_metric(df, dataset_name, "fgsm_accuracy")
    tables["examples_10_count"] = pivot_metric(df, dataset_name, "examples_10_count")
    tables["examples_01_count"] = pivot_metric(df, dataset_name, "examples_01_count")
    tables["examples_000_count"] = pivot_metric(df, dataset_name, "examples_000_count")
    tables["examples_001_count"] = pivot_metric(df, dataset_name, "examples_001_count")
    tables["examples_11_count"] = pivot_metric(df, dataset_name, "examples_11_count")

    # Optional derived rate tables
    derived = df.copy()
    for col in [
        "examples_10_count",
        "examples_01_count",
        "examples_000_count",
        "examples_001_count",
        "examples_11_count",
    ]:
        derived[col.replace("_count", "_rate")] = derived[col] / 10000.0

    tables["examples_10_rate"] = pivot_metric(derived, dataset_name, "examples_10_rate")
    tables["examples_01_rate"] = pivot_metric(derived, dataset_name, "examples_01_rate")
    tables["examples_000_rate"] = pivot_metric(derived, dataset_name, "examples_000_rate")
    tables["examples_001_rate"] = pivot_metric(derived, dataset_name, "examples_001_rate")
    tables["examples_11_rate"] = pivot_metric(derived, dataset_name, "examples_11_rate")

    return tables


# ============================================================
# DRIVER
# ============================================================

def main():
    files = sorted(LOG_DIR.glob("FGSM*.txt"))
    if not files:
        raise FileNotFoundError(f"No FGSM*.txt files found in {LOG_DIR.resolve()}")

    all_records = []
    for path in files:
        all_records.extend(parse_log_file(path))

    validate_all(all_records)

    df = pd.DataFrame(all_records)
    df = df.sort_values(["dataset", "model_variant", "epsilon"]).reset_index(drop=True)

    # Save raw parsed data
    df.to_csv(OUTPUT_DIR / "parsed_fgsm_records.csv", index=False)
    df.to_csv(OUTPUT_DIR / "parsed_fgsm_records.tsv", sep="\t", index=False)

    # CIFAR-10 first, then CIFAR-100
    for dataset_name in ["CIFAR-10", "CIFAR-100"]:
        tables = build_all_tables(df, dataset_name)

        print(f"\n==================== {dataset_name} ====================")
        for table_name, table_df in tables.items():
            title = f"{dataset_name} - {table_name}"
            print_tsv(title, table_df)

            safe_name = (
                f"{dataset_name.lower().replace('-', '_')}_{table_name}"
            )
            save_table(table_df, safe_name)

    print(f"\nSaved all tables to: {OUTPUT_DIR.resolve()}")

if __name__ == "__main__":
    main()

Dataset: CIFAR-10
Dataset: CIFAR-100
Dataset: CIFAR-100
Dataset: CIFAR-100
Dataset: CIFAR-100
Dataset: CIFAR-100
Dataset: CIFAR-10
Dataset: CIFAR-10
Dataset: CIFAR-10
Dataset: CIFAR-10

==================== CIFAR-10 ====================

CIFAR-10 - clean_accuracy
Model Variant	0.0078	0.0627
Dense	0.9155	0.9155
Pruned 20%	0.9185	0.9185
Pruned 40%	0.9186	0.9186
Pruned 60%	0.9198	0.9198
Pruned 80%	0.9161	0.9161


CIFAR-10 - fgsm_accuracy
Model Variant	0.0078	0.0627
Dense	0.0735	0.1458
Pruned 20%	0.0731	0.1475
Pruned 40%	0.0706	0.1433
Pruned 60%	0.0723	0.143
Pruned 80%	0.0727	0.1387


CIFAR-10 - examples_10_count
Model Variant	0.0078	0.0627
Dense	8421	7804
Pruned 20%	8455	7817
Pruned 40%	8482	7858
Pruned 60%	8476	7863
Pruned 80%	8434	7861


CIFAR-10 - examples_01_count
Model Variant	0.0078	0.0627
Dense	1	107
Pruned 20%	1	107
Pruned 40%	2	105
Pruned 60%	1	95
Pruned 80%	0	87


CIFAR-10 - examples_000_count
Model Variant	0.0078	0.0627
Dense	184	447
Pruned 20%	172	437
Pruned 40%	155	427
Pruned

/tmp/ipykernel_6951/798192221.py:172: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pt = sub.pivot_table(
/tmp/ipykernel_6951/798192221.py:172: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pt = sub.pivot_table(
/tmp/ipykernel_6951/798192221.py:172: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pt = sub.pivot_table(
/tmp/ipykernel_6951/798192221.py:172: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False 


==================== CIFAR-100 ====================

CIFAR-100 - clean_accuracy
Model Variant	0.0078	0.0627
Dense	0.7263	0.7263
Pruned 20%	0.7284	0.7284
Pruned 40%	0.7337	0.7337
Pruned 60%	0.7308	0.7308
Pruned 80%	0.7245	0.7245


CIFAR-100 - fgsm_accuracy
Model Variant	0.0078	0.0627
Dense	0.0435	0.0418
Pruned 20%	0.0391	0.0401
Pruned 40%	0.0449	0.0386
Pruned 60%	0.0433	0.0398
Pruned 80%	0.0418	0.0388


CIFAR-100 - examples_10_count
Model Variant	0.0078	0.0627
Dense	6828	6927
Pruned 20%	6893	6960
Pruned 40%	6888	7022
Pruned 60%	6875	6982
Pruned 80%	6827	6913


CIFAR-100 - examples_01_count
Model Variant	0.0078	0.0627
Dense	0	82
Pruned 20%	0	77
Pruned 40%	0	71
Pruned 60%	0	72
Pruned 80%	0	56


CIFAR-100 - examples_000_count
Model Variant	0.0078	0.0627
Dense	1338	2537
Pruned 20%	1278	2511
Pruned 40%	1300	2471
Pruned 60%	1257	2482
Pruned 80%	1312	2575


CIFAR-100 - examples_001_count
Model Variant	0.0078	0.0627
Dense	1399	118
Pruned 20%	1438	128
Pruned 40%	1363	121
Pruned 60%	1435	138
Pru

/tmp/ipykernel_6951/798192221.py:172: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pt = sub.pivot_table(
